In [10]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, KFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.model_selection import StratifiedKFold


In [11]:
seed = 42
time_for_eval = 24
quick = True

R, K = (1, 5) if quick else (5, 5)
n_trees = 200 if quick else 500

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

data = pd.read_csv("data/acrg_clean.csv")
features = ["age", "male", "serosa", "n_level"]
data["male"] = (data["sex"] == "Male").astype(float)
X = data[features].astype(float)
y = Surv.from_arrays(event=data["event"].astype(bool).values, time=data["time_months"].values)
print(X.shape, "events:", int(data["event"].sum()))

(252, 4) events: 103


In [12]:
def c_index_time_for_eval(y_true, risk, tfe = time_for_eval):
    event_tfe = y_true["event"] & (y_true["time"] < tfe)
    return concordance_index_censored(event_tfe, y_true["time"], risk)[0]

def eval_times(y_test, tfe = time_for_eval):
    hi = min(tfe, y_test["time"].max() - 1e-6)
    lo = max(1.0, y_test["time"].min() + 1e-6)
    return np.linspace(lo, hi, 24)

def surv_at(model, X_, times):
    return np.vstack([fn(times) for fn in model.predict_survival_function(X_)])

def km_null_surv(y_train, times, n):
    t, s = kaplan_meier_estimator(y_train["event"], y_train["time"])
    s_at = np.array([s[t <= u][-1] if (t <= u).any() else 1.0 for u in times])
    return np.tile(s_at, (n, 1))

In [13]:
def time_of_eval_scorer(estimator, X_, y_):
    return c_index_time_for_eval(y_, estimator.predict(X_))

def inner_cv(y_tr):
    return list(StratifiedKFold(3, shuffle=True, random_state=seed).split(np.zeros(len(y_tr)), y_tr["event"]))

def make_cox(y_tr):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("cox", CoxnetSurvivalAnalysis(fit_baseline_model=True))])
    grid = {"cox__alphas": [[1e-3], [1e-2], [1e-1], [1.0]], "cox__l1_ratio": [0.1, 0.5, 0.9]}
    return GridSearchCV(pipe, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer, error_score=0.5)

def make_rsf(y_tr):
    rsf = RandomSurvivalForest(n_estimators=n_trees, random_state=seed, n_jobs=-1)
    grid = {"min_samples_leaf": [5, 10, 20, 40], "max_features": [1, 2, 4]}
    return GridSearchCV(rsf, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer)

In [14]:
import copy
import numpy as np
import torch
from torch import nn
from sklearn.base import BaseEstimator
from sklearn.preprocessing import StandardScaler
from sksurv.functions import StepFunction
from sksurv.metrics import concordance_index_censored

In [15]:
class DeepSurv(BaseEstimator):
    def __init__(self, hidden_layers = 1, units = 16, dropout = 0.1, weight_decay = 1e-3, lr = 1e-3, max_epochs = 500, patience = 20, val_fraction = 0.2, n_seeds = 3, random_state = 0):
        self.hidden_layers = hidden_layers
        self.units = units
        self.dropout = dropout
        self.weight_decay = weight_decay
        self.lr = lr
        self.max_epochs = max_epochs
        self.patience = patience
        self.val_fraction = val_fraction
        self.n_seeds = n_seeds
        self.random_state = random_state

    def _build_net(self, n_inputs):
        layers, d = [], n_inputs
        for _ in range(self.hidden_layers):
            layers += [nn.Linear(d, self.units), nn.ReLU(), nn.Dropout(self.dropout)]
            d = self.units
        layers.append(nn.Linear(d, 1))
        return nn.Sequential(*layers)

    @staticmethod
    def _cox_loss(g, time, event):
        order = torch.argsort(time, descending=True)
        g, event = g[order], event[order]
        log_risk_set = torch.logcumsumexp(g, dim = 0)
        return -((g - log_risk_set) * event).sum() / event.sum().clamp(min = 1.0)

    def fit(self, X, y):
        X = np.asarray(X, dtype = np.float32)
        self.scaler_ = StandardScaler().fit(X)
        Xs = torch.tensor(self.scaler_.transform(X), dtype = torch.float32)
        time = torch.tensor(np.ascontiguousarray(y["time"], dtype=np.float32))
        event = torch.tensor(np.ascontiguousarray(y["event"], dtype=np.float32))

        rng = np.random.default_rng(self.random_state)
        ev = np.asanyarray(y["event"]).astype(bool)
        val = np.zeros(len(ev), dtype = bool)
        for grp in (np.where(ev)[0], np.where(~ev)[0]):
            k = max(1, int(round(self.val_fraction * len(grp))))
            val[rng.choice(grp, size = k, replace = False)] = True
        tr = ~val

        self.nets_ = []
        for s in range(self.n_seeds):
            torch.manual_seed(self.random_state + s)
            net = self._build_net(Xs.shape[1])
            opt = torch.optim.Adam(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            best_loss, best_state, wait = np.inf, None, 0

            for epoch in range(self.max_epochs):
                net.train()
                opt.zero_grad()
                loss = self._cox_loss(net(Xs[tr]).squeeze(1), time[tr], event[tr])
                loss.backward()
                opt.step()
                net.eval()
                with torch.no_grad():
                    val_loss = self._cox_loss(net(Xs[val]).squeeze(1), time[val], event[val]).item()
                if val_loss < best_loss - 1e-4:
                    best_loss, best_state, wait = val_loss, copy.deepcopy(net.state_dict()), 0
                else:
                    wait += 1
                    if wait >= self.patience:
                        break
            net.load_state_dict(best_state)
            net.eval()
            self.nets_.append(net)
        r = self._log_risk(X)
        t, e = np.asarray(y["time"], float), np.asarray(y["event"], bool)
        self.event_times_ = np.unique(t[e])
        exp_r = np.exp(r)
        H0 = np.cumsum([e[t == u].sum() / exp_r[t >= u].sum() for u in self.event_times_])
        self.baseline_cumhaz_ = H0
        return self

    def _log_risk(self, X):
        Xs = torch.tensor(self.scaler_.transform(np.asarray(X, dtype = np.float32)), dtype = torch.float32)
        with torch.no_grad():
            return np.mean([net(Xs).squeeze(1).numpy() for net in self.nets_], axis=0)

    def predict(self, X):
        return self._log_risk(X)

    def predict_survival_function(self, X):
        r = self._log_risk(X)
        x = np.r_[0.0, self.event_times_]
        return [StepFunction(x, np.exp(-np.r_[0.0, self.baseline_cumhaz_] * np.exp(ri))) for ri in r]

    def score(self, X, y):
        return concordance_index_censored(y["event"], y["time"], self.predict(X))[0]

In [16]:
def make_deepsurv(y_tr):
    net = DeepSurv(n_seeds = 3, random_state = seed)
    grid = {"hidden_layers": [1, 2], "units": [8, 32], "dropout": [0.1, 0.3], "weight_decay": [1e-4, 1e-2]}
    return GridSearchCV(net, grid, cv=inner_cv(y_tr), scoring=time_of_eval_scorer)

models = {"Cox": make_cox, "RSF": make_rsf, "DeepSurv": make_deepsurv}

In [17]:
rows, out_of_fold = [], []
cv = RepeatedStratifiedKFold(n_splits=K, n_repeats=R, random_state=seed)
t0 = time.time()

for split, (tr, te) in enumerate (cv.split(X, y["event"])):
    rep, fold = divmod(split, K)
    X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]
    times = eval_times(y_te)

    risk0 = (X_te["serosa"] + X_te["n_level"]).values
    rows.append({"rep": rep, "fold": fold, "model": "Stage only", "c_tau": c_index_time_for_eval(y_te, risk0), "ibs": np.nan, "params": ""})
    rows.append({"rep": rep, "fold": fold, "model": "KM (no predictors)", "c_tau": 0.5,
                 "ibs": integrated_brier_score(y_tr, y_te, km_null_surv(y_tr, times, len(te)), times), "params": ""})

    for name, make in models.items():
        search = make(y_tr).fit(X_tr, y_tr)
        best = search.best_estimator_
        risk = best.predict(X_te)
        S = surv_at(best, X_te, times)
        rows.append({"rep": rep, "fold": fold, "model": name, "c_tau": c_index_time_for_eval(y_te, risk),
                     "ibs": integrated_brier_score(y_tr, y_te, S, times), "params": str(search.best_params_)})
        out_of_fold.append(pd.DataFrame({"rep": rep, "model": name, "patient_id": data["patient_id"].iloc[te].values,
                                 "risk": risk, "surv_tau": S[:, -1]}))
    print(f"split {split+1}/{R*K} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
out_of_fold = pd.concat(out_of_fold, ignore_index=True)
res.to_csv(results_dir / "e1_acrg_folds.csv", index=False)
out_of_fold.to_csv(results_dir / "e1_acrg_oof.csv", index=False)

C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 1/5 done (91s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 2/5 done (200s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 3/5 done (298s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 4/5 done (421s)


C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
C:\Personal\AllCollegeStuff\Semester 5\Cercetare\GastricCancerRecurrence\.venv312\Lib\site-packages\sklearn\pipeline.py:645: UserWarning: all coefficients are zero, consider decreasing alpha.
  self._final_estimator.fit(Xt, y, **last_step

split 5/5 done (516s)


In [18]:
summary = res.groupby("model")[["c_tau", "ibs"]].agg(["mean", "std"]).round(3)
display(summary)

def corrected_ttest(d, n_train, n_test, alpha=0.05):
    d = np.asarray(d, float); J = len(d)
    se = np.sqrt((1 / J + n_test / n_train) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=J - 1)
    q = stats.t.ppf(1 - alpha / 2, df=J - 1)
    return d.mean(), d.mean() - q * se, d.mean() + q * se, p

n_test = len(X) // K; n_train = len(X) - n_test
wide = res.pivot_table(index=["rep", "fold"], columns="model", values="c_tau")
EPS = 0.05
for m in ["RSF", "DeepSurv", "Stage only"]:
    d = wide[m] - wide["Cox"]
    mean, lo95, hi95, p = corrected_ttest(d, n_train, n_test)
    _, lo90, hi90, _ = corrected_ttest(d, n_train, n_test, alpha=0.10)
    equiv = (lo90 > -EPS) and (hi90 < EPS)
    print(f"{m} - Cox: {mean:+.3f}  95% CI [{lo95:+.3f}, {hi95:+.3f}]  p={p:.2f}  equivalent within ±{EPS}: {equiv}")

c_tau           ibs       
                     mean    std   mean    std
model                                         
Cox                 0.680  0.056  0.146  0.018
DeepSurv            0.682  0.052  0.147  0.016
KM (no predictors)  0.500  0.000  0.163  0.017
RSF                 0.683  0.066  0.147  0.019
Stage only          0.686  0.062    NaN    NaN

RSF - Cox: +0.003  95% CI [-0.033, +0.038]  p=0.84  equivalent within ±0.05: True
DeepSurv - Cox: +0.001  95% CI [-0.010, +0.012]  p=0.77  equivalent within ±0.05: True
Stage only - Cox: +0.005  95% CI [-0.035, +0.046]  p=0.73  equivalent within ±0.05: True
